In [ ]:
import numpy as np
from numpy import array
import seaborn as sns
import pandas as pd
from statsmodels.stats.power import FTestAnovaPower
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from matplotlib import pyplot

In [ ]:
# Import dataset (we transform the xlsv to csv for convenience)
df = pd.read_csv("../data/project2_data.csv")

# Data Exploration


In [ ]:
# Data Exploration
# Total number of rows
print("Total rows:", len(df))

# Missing values (only columns with at least 1 missing value)
missing = df.isnull().sum()
print("\nMissing values by column:")
print(missing)

Total rows: 214

Missing values by column:
ID                0
Age              56
COLORAM           0
COLORPM           1
Comments_COG    156
Comments_Sch      0
dehydAM           0
dehydPM           0
EAUJOUR2         94
Grade             0
Intervention      0
LCS               0
LCC               0
LCI               0
LCX             211
IDC               0
IDI               0
IDX             206
IDS               0
IIC               0
III               0
IIX             204
IIS               0
NFC               0
NFM               0
NFX             214
NRC               0
NRM               0
NRX             180
LT                4
LTM               4
LTE               4
LTX             195
P105              0
P106              0
P107              0
P108             15
Pres1             0
Pres2             0
school            0
sex              42
SGAM              0
SGPM              0
Test1             0
Test2             0
Thirsty           2
Temp1            94
Humid1           

# Experimental Design

## Hypothesis 1 - Crossover Design - Terrence

Hypothesis: Water intervention improves short-term memory performance as measured by Number Recall Reverse scores (NRC), when controlling for visit order effects.


### Blocking Only Analysis

In [ ]:
# Subset relevant data (No missing values detected)
df_block1 = df[['NRC', 'water', 'visit', 'school']].copy()

# Two-way ANOVA
print("\nANOVA Results for Hypothesis 1 (Blocking Only):")
model1_block = smf.ols('NRC ~ C(water) * C(visit) + C(school)', data=df_block1).fit()
print(sm.stats.anova_lm(model1_block, typ=2).round(3))

# Post-hoc test
df_block1['group'] = df_block1['water'].astype(str) + "_" + df_block1['visit'].astype(str)
print("\nPost-hoc Tukey Test Results for Hypothesis 1 (Blocking Only):")
tukey1_block = pairwise_tukeyhsd(endog=df_block1['NRC'], groups=df_block1['group'], alpha=0.05)
print(tukey1_block)



ANOVA Results for Hypothesis 1 (Blocking Only):
                    sum_sq     df      F  PR(>F)
C(water)            21.141    1.0  7.469   0.007
C(visit)             0.333    1.0  0.118   0.732
C(school)           17.069    1.0  6.031   0.015
C(water):C(visit)    1.791    1.0  0.633   0.427
Residual           594.375  210.0    NaN     NaN

Post-hoc Tukey Test Results for Hypothesis 1 (Blocking Only):
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
   0_1    0_2   0.6323 0.2189 -0.2164 1.4809  False
   0_1    1_1   1.0656 0.0073  0.2169 1.9143   True
   0_1    1_2   0.7872 0.1088 -0.1115  1.686  False
   0_2    1_1   0.4333  0.494 -0.3621 1.2288  False
   0_2    1_2    0.155  0.965 -0.6937 1.0036  False
   1_1    1_2  -0.2784 0.8307  -1.127 0.5703  False
---------------------------------------------------


### Covariate Only Analysis

In [ ]:
# Subset relevant data (No missing values detected)
df_cov1 = df[['NRC', 'water', 'visit', 'SGAM']].copy()

# Two-way ANOVA
print("\nANOVA Results for Hypothesis 1 (Covariate Only):")
model1_cov = smf.ols('NRC ~ C(water) * C(visit) + SGAM', data=df_cov1).fit()
print(sm.stats.anova_lm(model1_cov, typ=2).round(3))

# Post-hoc test
df_cov1['group'] = df_cov1['water'].astype(str) + "_" + df_cov1['visit'].astype(str)
print("\nPost-hoc Tukey Test Results for Hypothesis 1 (Covariate Only):")
tukey1_cov = pairwise_tukeyhsd(endog=df_cov1['NRC'], groups=df_cov1['group'], alpha=0.05)
print(tukey1_cov)


ANOVA Results for Hypothesis 1 (Covariate Only):
                    sum_sq     df      F  PR(>F)
C(water)            19.049    1.0  6.701   0.010
C(visit)             1.620    1.0  0.570   0.451
C(water):C(visit)    9.994    1.0  3.516   0.062
SGAM                 0.275    1.0  0.097   0.756
Residual           594.100  209.0    NaN     NaN

Post-hoc Tukey Test Results for Hypothesis 1 (Covariate Only):
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
   0_1    0_2   0.6323 0.2189 -0.2164 1.4809  False
   0_1    1_1   1.0656 0.0073  0.2169 1.9143   True
   0_1    1_2   0.7872 0.1088 -0.1115  1.686  False
   0_2    1_1   0.4333  0.494 -0.3621 1.2288  False
   0_2    1_2    0.155  0.965 -0.6937 1.0036  False
   1_1    1_2  -0.2784 0.8307  -1.127 0.5703  False
---------------------------------------------------


## Hypothesis 2 - Factorial Design

Hypothesis: Water intervention and morning eating status jointly influence visual attention performance, as measured by Letter Cancellation scores (LCS). We hypothesize that both drinking water and eating breakfast positively affect cognitive performance by solving hydration and providing necessary energy, respectively.
We further expect an interaction effect: The impact of providing water on students’ LCS scores may vary depending on whether students ate breakfast or not, as those who ate may be better primed to benefit cognitively from hydration, potentially amplifying the effect of the water intervention compared to students who skipped breakfast.


### Blocking Only Analysis

In [ ]:
# Subset relevant data (No missing values detected)
df_block2 = df[['LCS', 'water', 'P106', 'dehydAM']].copy()

# Two-way ANOVA
print("\nANOVA Results for Hypothesis 2 (Blocking Only):")
model2_block = smf.ols('LCS ~ C(water) * C(P106) + C(dehydAM)', data=df_block2).fit()
print(sm.stats.anova_lm(model2_block, typ=2).round(3))

# Post-hoc test
df_block2['group'] = df_block2['water'].astype(str) + "_" + df_block2['P106'].astype(str)
print("\nPost-hoc Tukey Test Results for Hypothesis 2 (Blocking Only):")
tukey2_block = pairwise_tukeyhsd(endog=df_block2['LCS'], groups=df_block2['group'], alpha=0.05)
print(tukey2_block)


ANOVA Results for Hypothesis 2 (Blocking Only):
                     sum_sq     df      F  PR(>F)
C(water)             12.609    1.0  0.185   0.667
C(P106)               7.631    1.0  0.112   0.738
C(dehydAM)            2.528    1.0  0.037   0.847
C(water):C(P106)     59.899    1.0  0.881   0.349
Residual          14209.106  209.0    NaN     NaN

Post-hoc Tukey Test Results for Hypothesis 2 (Blocking Only):
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
   0_0    0_1   0.4427 0.9979  -5.8614  6.7468  False
   0_0    1_0   3.7885 0.8518  -8.3927 15.9696  False
   0_0    1_1  -0.3062 0.9993  -6.5767  5.9643  False
   0_1    1_0   3.3457 0.8559  -7.5306 14.2221  False
   0_1    1_1  -0.7489 0.9195  -3.7878    2.29  False
   1_0    1_1  -4.0947 0.7629 -14.9516  6.7623  False
-----------------------------------------------------


### Covariate Only Analysis

In [ ]:
# Power analysis for factorial ANOVA design
# Define parameters for power analysis
effect_size = 0.5  # Medium effect size (Cohen's f)
alpha = 0.05        # Significance level
power = 0.8         # Desired power
num_groups = 4    # 2x2 factorial design (2 factors with 2 levels each)

# Create power analysis object
power_analysis = FTestAnovaPower()

# Calculate required sample size
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    nobs=None,
    alpha=alpha,
    power=power,
    k_groups=num_groups
)

# Print result
print(f"Required total sample size for 2x2 factorial design: {round(sample_size)}")

Required total sample size for 2x2 factorial design: 48


In [ ]:
# Function to evaluate if sufficient data remains after excluding missing age values
def evaluate_sample_sufficiency(df, required_size):
    total_rows = len(df)
    missing_age_count = df['Age'].isna().sum()
    remaining_rows = total_rows - missing_age_count

    print(f"Total observations: {total_rows}")
    print(f"Missing age values: {missing_age_count}")
    print(f"Remaining observations after exclusion: {remaining_rows}")
    print(f"Required observations for adequate power: {required_size}")

    if remaining_rows > required_size:
        print("The results inform our decision to exclude cases with missing age data while "
              "maintaining enough statistical power for our analyses.")
    else:
        print("WARNING: Excluding cases with missing age data would result in insufficient "
              "statistical power. Consider imputation or alternative analytical approaches.")

# Run the function
evaluate_sample_sufficiency(df, round(sample_size))

Total observations: 214
Missing age values: 56
Remaining observations after exclusion: 158
Required observations for adequate power: 48
The results inform our decision to exclude cases with missing age data while maintaining enough statistical power for our analyses.


In [ ]:
# Subset relevant data and data cleaning (Missing values detected)
df_cov2 = df[['LCS', 'water', 'P106', 'Age']].dropna()

# Two-way ANOVA
print("\nANOVA Results for Hypothesis 2 (Covariate Only):")
model2_cov = smf.ols('LCS ~ C(water) * C(P106) + Age', data=df_cov2).fit()
print(sm.stats.anova_lm(model2_cov, typ=2).round(3))

# Post-hoc test
df_cov2['group'] = df_cov2['water'].astype(str) + "_" + df_cov2['P106'].astype(str)
print("\nPost-hoc Tukey Test for Hypothesis 2 (Covariate Only):")
tukey2_cov = pairwise_tukeyhsd(endog=df_cov2['LCS'], groups=df_cov2['group'], alpha=0.05)
print(tukey2_cov)


ANOVA Results for Hypothesis 2 (Covariate Only):
                    sum_sq     df      F  PR(>F)
C(water)           277.581    1.0  5.059   0.026
C(P106)             55.618    1.0  1.014   0.316
C(water):C(P106)   108.582    1.0  1.979   0.162
Age                  5.992    1.0  0.109   0.742
Residual          8395.136  153.0    NaN     NaN

Post-hoc Tukey Test for Hypothesis 2 (Covariate Only):
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
   0_0    0_1  12.8974 0.3091  -6.4088 32.2037  False
   0_0    1_0     10.0 0.6867 -13.4951 33.4951  False
   0_0    1_1   10.026 0.5335  -9.2818 29.3338  False
   0_1    1_0  -2.8974  0.947 -16.6351 10.8403  False
   0_1    1_1  -2.8715 0.0775  -5.9533  0.2103  False
   1_0    1_1    0.026    1.0  -13.714 13.7659  False
-----------------------------------------------------
